# AutoGen — Multi-Agent Conversation
## Week 2 Day 3 (05-13): Dialogue Abstraction for Multi-Agent Collaboration

**Core Goal**: Run AutoGen official quickstart, understand multi-agent conversation patterns.

**Key difference from LangGraph**:
- **LangGraph** (engineer's view): Explicit state machine (Node + Edge), fine-grained control
- **AutoGen** (researcher's view): Agents collaborate through natural language dialogue, emergent behavior


## 1. Environment Setup
Import AutoGen core modules + load API Key.

In [ ]:
import asyncio, json, math, os, sys
from dotenv import load_dotenv
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

load_dotenv()

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

print("AutoGen imports OK")


## 2. Model Client Factory
AutoGen 0.7 uses `OpenAIChatCompletionClient` as a unified LLM interface for OpenAI-compatible APIs (including DeepSeek).

In [ ]:
def make_model_client():
    return OpenAIChatCompletionClient(
        model="deepseek-v4-flash",
        api_key=os.getenv("API_KEY"),
        base_url="https://api.deepseek.com/v1",
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": False,
            "family": "deepseek",
        },
    )

client = make_model_client()
print("Model client created")


## 3. Core Concept: SelectorGroupChat

**AutoGen's key ideas**:
1. **Dialogue abstraction**: Agents communicate via natural language messages, not explicit state transitions
2. **SelectorGroupChat**: LLM acts as "moderator" dynamically choosing the next speaker
3. **Emergent behavior**: Collaboration patterns emerge from conversation, not pre-defined workflows

**Three abstraction philosophies (Interview Q25)**:
- `LangGraph` -> Graph abstraction -> "I know what the workflow is"
- `AutoGen` -> Dialogue abstraction -> "Let agents figure it out themselves"
- `CrewAI` -> Role abstraction -> "Everyone has their own job"


## 4. Demo 1: Two-Agent Dialogue (asker vs expert)

**SelectorGroupChat** manages turns automatically, LLM decides who speaks.

This simplest example shows AutoGen's core pattern:
- Two `AssistantAgent` instances converse with each other
- LLM selector chooses the next speaker
- `TextMentionTermination` stops when specific text is detected

In [ ]:
async def basic_demo():
    client = make_model_client()

    asker = AssistantAgent(
        name="asker",
        model_client=client,
        system_message="You are 'asker', a developer curious about AI Agent tech. Ask ONE question in Chinese about LangGraph vs AutoGen vs CrewAI framework selection. Just ask the question, nothing else.",
    )

    expert = AssistantAgent(
        name="expert",
        model_client=client,
        system_message="You are 'expert', an AI architect. Answer in Chinese concisely (3-5 sentences). Start with conclusion then explain. End your response with 'DONE.'",
    )

    team = SelectorGroupChat(
        participants=[asker, expert],
        model_client=client,
        selector_prompt="First pick asker (to ask), then pick expert (to answer), then pick TERMINATE.",
        termination_condition=TextMentionTermination(text="DONE"),
    )

    result = await team.run(task="Start a dialogue. asker asks first, expert answers.")

    print("-" * 50)
    for msg in result.messages:
        if hasattr(msg, "source") and hasattr(msg, "content"):
            print(f"[{msg.source}]: {msg.content}")
            print()

    await client.close()

await basic_demo()


## 5. Demo 2: Three-Agent Collaboration (Researcher + Writer + Critic)

This is AutoGen's most valuable scenario: **multi-role division of labor through natural language**.

Three agents collaborate to produce a short article:
1. **Researcher** -> Provides facts and arguments
2. **Writer** -> Crafts the article from research material
3. **Critic** -> Reviews and suggests improvements

This pattern mirrors Project 3 (multi-agent-collab).

In [ ]:
async def group_demo():
    client = make_model_client()

    researcher = AssistantAgent(
        name="researcher", model_client=client,
        system_message="You are 'researcher'. Given a topic, provide 3-5 specific facts/arguments. End with 'RESEARCH_DONE'.",
    )
    writer = AssistantAgent(
        name="writer", model_client=client,
        system_message="You are 'writer'. Based on research, write a 150-200 word article (title + body + summary). End with 'WRITING_DONE'.",
    )
    critic = AssistantAgent(
        name="critic", model_client=client,
        system_message="You are 'critic'. Give 2-3 specific improvement suggestions. End with 'REVIEW_DONE_STOP'.",
    )

    team = SelectorGroupChat(
        participants=[researcher, writer, critic],
        model_client=client,
        selector_prompt="Order: researcher -> writer -> critic -> TERMINATE. Do NOT repeat agents.",
        termination_condition=TextMentionTermination(text="STOP"),
        max_turns=5,
    )

    result = await team.run(
        task="Collaborate on a short article: 'Why AI Agents Need Tool Calling'"
    )

    for msg in result.messages:
        if hasattr(msg, "source") and hasattr(msg, "content"):
            c = msg.content if isinstance(msg.content, str) else str(msg.content)[:300]
            print(f"[{msg.source}]: {c}")
            print()

    await client.close()

await group_demo()


## 6. Framework Comparison Summary (Interview Q25 Complete Answer)

| Dimension | LangGraph | AutoGen | CrewAI |
|-----------|-----------|---------|--------|
| **Core Abstraction** | Graph | Dialogue | Role |
| **Philosophy** | Engineer - explicit FSM | Researcher - emergent | Business - role modeling |
| **Controllability** | Highest | Medium | Medium |
| **Learning Curve** | Steepest | Medium | Easiest |
| **HITL Support** | Native | Supported | Limited |
| **Best For** | Production, fine control | Multi-agent research, prototypes | Business demos, teaching |

### Selection Guide (for interviews)

1. **Production core Agent** -> LangGraph (fine control + checkpoint + tracing)
2. **Multi-agent research/prototype** -> AutoGen (emergent dialogue + quick validation)
3. **Business demo/teaching** -> CrewAI (intuitive roles + fastest setup)
4. Follow Anthropic principle: **Start simple, add complexity only when needed**

### This Week's Hands-on Summary

| Day | Framework | Key Takeaway |
|-----|-----------|---------------|
| w2d1 | LangGraph | Define flow as a graph, precise but concept-heavy |
| w2d2 | LangGraph HITL | interrupt_before = one line for human approval |
| w2d3 | AutoGen | Let agents decide who speaks, emergent collaboration |

**Conclusion**: Graph abstraction works when you know the flow; dialogue abstraction works when you want agents to negotiate. They complement each other. Based on Anthropic's guidance and my own experience, for production systems I'd use LangGraph for the main orchestration and borrow AutoGen's conversation patterns when multi-agent negotiation is truly needed. But most of the time, a well-designed single agent with good tools is enough.
